# TP Vision Transformer

In this practical session, we explore various elements of the Vision Transformer (ViT) implementation in the TIMM library.

**Credits:**
- This notebook was prepared by Matteo Bastico, PhD student at Mines Paris : https://github.com/matteo-bastico, based on the notebook from Hiroto Honda: https://colab.research.google.com/github/hirotomusiker/schwert_colab_data_storage/blob/master/notebook/Vision_Transformer_Tutorial.ipynb.
- The ViT studied here was originally introduced in the Paper: *Alexey Dosovitskiy et al., "An Image Is Worth 16x16 Words: Transformers for Image Recognition at Scale* https://arxiv.org/abs/2010.11929
- Model Implementation: this notebook loads (and is inspired by) Ross Wightman (@wightmanr)'s module:
  https://github.com/rwightman/pytorch-image-models/tree/master/timm . For the detailed codes, please refer to the repo
- The copyright of figures and demo images belongs to Hiroto Honda.

In [ ]:
%%capture
!pip install timm

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import PIL

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

from timm import create_model
#torch.set_grad_enabled(False)  #  inférence uniquement

## Prepare Model and Data

Here, we directly create a ViT model by loading a pretrained model from timm module.

In [ ]:
#model_name = "vit_base_patch16_224"
model_name = "vit_base_patch16_224.augreg2_in21k_ft_in1k"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device = ", device)
model = create_model(model_name, pretrained=True).to(device)


<img src='https://machinelearningmastery.com/wp-content/uploads/2022/03/dotproduct_1.png'>



In [ ]:
print(model)

In [ ]:
print("NUM OF HEADS",model.blocks[0].attn.num_heads)
print("HEAD Dimension",model.blocks[0].attn.head_dim)

**Q1: Observe the previously printed model and variables. Respond to the sub-questions below:**
- How many attention blocks does the model have?
- What is the patch size?
- How many rows of patches are there ?
- How many columns of patches are there ?
- What is the dimension of the latent space?
- What is the dimension of the hidden layer of the MLP?
- The number of attention heads is 12 and the dimension of each head is 64. Is it correct? How do you explain it?
- What is the purpose of having several heads instead of just one ?

In [ ]:
# Define transforms for test : the image range is set to [-1, 1]
IMG_SIZE = (224, 224)
NORMALIZE_MEAN = (0.5, 0.5, 0.5)
NORMALIZE_STD = (0.5, 0.5, 0.5)
transforms = [
    T.Resize(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
]
transforms = T.Compose(transforms)

In [ ]:
%%capture

# Get ImageNet Labels
#!wget https://storage.googleapis.com/bit_models/ilsvrc2012_wordnet_lemmas.txt
!wget https://raw.githubusercontent.com/piotrhm/dl-miniproject/refs/heads/main/ilsvrc2012_wordnet_lemmas.txt

imagenet_labels = dict(enumerate(open('ilsvrc2012_wordnet_lemmas.txt')))

# Load a Demo Image
!wget https://github.com/hirotomusiker/schwert_colab_data_storage/blob/master/images/vit_demo/santorini.png?raw=true -O santorini.png
img = PIL.Image.open('santorini.png')
img_tensor = transforms(img).unsqueeze(0).to(device)  # unsqueeze to add batch size (1)
img_np = np.asarray(img.resize(IMG_SIZE))  # image à la taille réellement vue par le

## Simple Inference

Here, we perform a simple inference on an image

In [ ]:
# end-to-end inference
output = model(img_tensor)

In [ ]:
print("Inference Result:")
print(imagenet_labels[int(torch.argmax(output))])
plt.imshow(img)

## Dig into Vision Transformer

<img src='https://github.com/hirotomusiker/schwert_colab_data_storage/blob/master/images/vit_demo/vit_input.png?raw=true'>
Let's look at the details of the Vision Transformer!

The Vision Transformer inference pipeline is described below :

1. Split Image into Patches  
The input image is split into 14 x 14 vectors with dimension of 768 by Conv2d (k=16x16) with stride=(16, 16).
2. Add Position Embeddings  
Learnable position embedding vectors are added to the patch embedding vectors and fed to the transformer encoder.
3. Transformer Encoder  
The embedding vectors are encoded by the transformer encoder. The dimension of input and output vectors are the same.
4. MLP (Classification) Head  
The 0th output from the encoder is fed to the MLP head for classification to output the final classification results.


### 1. Split Image into Patches

**Q2: How do you think the image is split into patches? Suggestion: check the patch_embed Module of the model.**

# 1. Split Image into Patches

The input image is split into N patches (N = 14 x 14 for ViT-Base)
and converted to D=768 embedding vectors by learnable 2D convolution:
```
Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))

In [ ]:
patches = model.patch_embed(img_tensor)  # patch embedding convolution
print("Image tensor: ", img_tensor.shape)
print("Patch embeddings: ", patches.shape)

In [ ]:
# Patches visualization
fig = plt.figure(figsize=(8, 8))
fig.suptitle("Visualization of Patches", fontsize=24)
img = np.asarray(img)
for i in range(0, 196):
    x = i % 14
    y = i // 14
    patch = img[y*16:(y+1)*16, x*16:(x+1)*16]
    ax = fig.add_subplot(14, 14, i+1)
    ax.axes.get_xaxis().set_visible(False)
    ax.axes.get_yaxis().set_visible(False)
    ax.imshow(patch)

# 2. Position Embeddings
To make patches position-aware, learnable 'position embedding' vectors are added to the patch embedding vectors. The position embedding vectors learn distance within the image thus neighboring ones have high similarity.


In [ ]:
# Visualization of position embeddings
pos_embed = model.pos_embed
print(pos_embed.shape)

**Q3 : Why there are 197 embeddings in the model?**

**Q4 : Using Mathematical variables only, write the number of tokens in a Vision Transformer as a function of image height, width and patch size.**


Traditional sinusoidal embeddings can be understood in terms of cosine similarity: https://www.learndatasci.com/glossary/cosine-similarity/

<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg">

**Q5: What is represented in this figure? Try to understand the notion of similarity here**


In [ ]:
# Visualize position embedding similarities.
# One cell shows cos similarity between an embedding and all the other embeddings.
cos = torch.nn.CosineSimilarity(dim=1, eps=1e-6)
fig = plt.figure(figsize=(8, 8))
fig.suptitle("Visualization of position embedding similarities", fontsize=14)
for i in range(1, pos_embed.shape[1]):
    sim = F.cosine_similarity(pos_embed[0, i:i+1], pos_embed[0, 1:], dim=1)
    sim = sim.reshape((14, 14)).detach().cpu().numpy()
    ax = fig.add_subplot(14, 14, i)
    ax.axes.get_xaxis().set_visible(False)
    ax.axes.get_yaxis().set_visible(False)
    ax.imshow(sim)

**Q6. Do you think this interpretation in terms of cosine similarity applies to positional embeddings in the ViT ?**

**Q7. Using Course Contents and Online Search, list 3 types of positional embeddings, different from sinusoidal embeddings, that are typically used in Vision Transformers. For each type, write one phrase of explanation with your own words.**

### Make Transformer Input
A learnable class token is prepended to the patch embedding vectors as the 0th vector. Positional embeddings are then added to the input tokens.


**Q8. What is the role of the class token?**


In [ ]:
transformer_input = torch.cat((model.cls_token, patches), dim=1) + pos_embed
print("Transformer input: ", transformer_input.shape)

# 3. Transformer Encoder
<img src='https://github.com/hirotomusiker/schwert_colab_data_storage/blob/master/images/vit_demo/transformer_encoder.png?raw=true'>

Detailed schematic of Transformer Encoder.
- N (=197) embedded vectors are fed to the L (=12) series encoders.
- The vectors are divided into query, key and value after expanded by an fc layer.
- q, k and v are further divided into H (=12) and fed to the parallel attention heads.
- Outputs from attention heads are concatenated to form the vectors whose shape is the same as the encoder input.
- The vectors go through an fc, a layer norm and an MLP block that has two fc layers.

The Vision Transformer employs the Transformer Encoder that was proposed in the [attention is all you need paper](https://papers.nips.cc/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf).

<img src="https://miro.medium.com/v2/resize:fit:814/1*SikzJ5iL95KcOr5tNzvOmQ.png">

Implementation Reference:

- [tensorflow implementation](https://github.com/google-research/vision_transformer/blob/502746cb287a107f9911c061f9d9c2c0159c81cc/vit_jax/models.py#L62-L146)
- [pytorch implementation (timm)](https://github.com/rwightman/pytorch-image-models/blob/198f6ea0f3dae13f041f3ea5880dd79089b60d61/timm/models/vision_transformer.py#L79-L143)


### Series Transformer Encoders

In [ ]:
print("Input tensor to Transformer (z0): ", transformer_input.shape)
x = transformer_input.clone()
for i, blk in enumerate(model.blocks):
    print("Entering the Transformer Encoder {}".format(i))
    x = blk(x)
x = model.norm(x)
transformer_output = x[:, 0]
print("Output vector from Transformer (z12-0):", transformer_output.shape)


## How Attention Works

In this part, we are going to see what the actual attention looks like.

**Q9: Complete the two missing lines in the code below in order to visualize the attention of the CLS token for each attention head of the first transformer block.**

In [ ]:

torch.set_grad_enabled(False)  # inférence uniquement
def block_attention(blk, x):
    """Attention weights (B, H, N, N) of block `blk` for its input x (B, N, D),
    calculated as in timm."""
    attn = blk.attn
    xn = blk.norm1(x)                                    # LayerNorm pré-attention
    B, N, _ = xn.shape
    qkv = attn.qkv(xn).reshape(B, N, 3, attn.num_heads, -1).permute(2, 0, 3, 1, 4)
    q, k = attn.q_norm(qkv[0]), attn.k_norm(qkv[1])     # (B, H, N, D/H) ; Identity pour ce modèle
    scores = (q @ k.transpose(-2, -1)) * attn.scale      # facteur 1/sqrt(D/H)
    return scores.softmax(dim=-1)


def all_attentions(model, transformer_input):
    """All layer attention weights, in a single pass."""
    x = transformer_input
    attns = []
    for blk in model.blocks:
        attns.append(block_attention(blk, x))
        x = blk(x)
    return attns  # liste de L tenseurs (B, H, N, N)

attns = all_attentions(model, transformer_input)
attention_matrix = #TODO
print("attention matrix: ", attention_matrix.shape)

# Verification: compare computed attention weights with those produced by the timm module.
blk0 = model.blocks[0]
xn = blk0.norm1(transformer_input)
v = blk0.attn.qkv(xn).reshape(1, 197, 3, 12, 64)[:, :, 2].transpose(1, 2)  # (1, H, N, 64)
out = blk0.attn.proj((attns[0] @ v).transpose(1, 2).reshape(1, 197, 768))
print("Maximum difference from timm :", (out - blk0.attn(xn)).abs().max().item())  # expected : ~1e-6

# Visualisation : QUERY row of the attention matrix (layer 0), heads 0 to 6
QUERY = #TODO
qy, qx = divmod(QUERY - 1, 14)    # position of the query patch in the grid 14 x 14

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle(f"Layer 0 : attention of token {QUERY} (red squqre), heads 0 to 6", fontsize=18)
rows = [(attention_matrix, "model : softmax(q·kᵀ/√d)")]
for r, (A, name) in enumerate(rows):
    axes[r, 0].imshow(img_np)
    axes[r, 0].add_patch(plt.Rectangle((qx * 16, qy * 16), 16, 16, fill=False, ec="red", lw=2))
    axes[r, 0].set_title(name, fontsize=10)
    for h in range(7):
        heat = A[h, QUERY, 1:].reshape(14, 14).cpu().numpy()
        axes[r, h + 1].imshow(heat)
        axes[r, h + 1].add_patch(plt.Rectangle((qx - .5, qy - .5), 1, 1, fill=False, ec="red"))
        axes[r, h + 1].set_title(f"tête {h}")
for ax in axes.ravel():
    ax.axis("off")
plt.show()


**Q10: Copy paste the previous block of code and modify it to display the visualization of attention heads of the 99th patch.**

# 4. MLP (Classification) Head
The 0-th output vector from the transformer output vectors (corresponding to the class token input) is fed to the MLP head.  
The 1000-dimension classification result is the output of the whole pipeline.


In [ ]:
print("Classification head: ", model.head)
result = model.head(transformer_output)
result_label_id = int(torch.argmax(result))
plt.plot(result.detach().cpu().numpy()[0])
plt.title("Classification result")
plt.xlabel("class id")
print("Inference result : id = {}, label name = {}".format(
    result_label_id, imagenet_labels[result_label_id]))

**Q11 : Display the 10 top predictions, from least to most likely.**